In [ ]:
def get_pending_periods(queue_df):
    return [
        row["rail_period"]
        for row in queue_df.select("rail_period")
                           .distinct()
                           .collect()
    ]

In [ ]:
def load_table(df, table_name):    
    (
        df.write.mode("append").synapsesql(
            table_name
        )
    )

In [ ]:
def execute_sql(sql_text, WAREHOUSE_SERVER, WAREHOUSE):

    token = mssparkutils.credentials.getToken(
        "https://database.windows.net/"
    )

    # Convert the Entra token into the format expected by ODBC
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack("<I", len(token_bytes)) + token_bytes

    connection_string = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server=tcp:{WAREHOUSE_SERVER},1433;"
        f"Database={WAREHOUSE};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
    )

    with pyodbc.connect(
            connection_string,
            attrs_before={1256: token_struct},
            autocommit=False
        ) as conn:
        cursor = conn.cursor()

        cursor.execute(sql_text)            
            
        conn.commit()
        cursor.close()

In [ ]:
def delete_data_by_rail_period(periods, WAREHOUSE_SERVER, WAREHOUSE):    
    token = mssparkutils.credentials.getToken(
        "https://database.windows.net/"
    )

    # Convert the Entra token into the format expected by ODBC
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack("<I", len(token_bytes)) + token_bytes

    connection_string = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server=tcp:{WAREHOUSE_SERVER},1433;"
        f"Database={WAREHOUSE};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
    )

    with pyodbc.connect(
            connection_string,
            attrs_before={1256: token_struct},
            autocommit=False
        ) as conn:
        cursor = conn.cursor()

        for rail_period in periods:
            print(f"Deleting data for : {rail_period}")        
            cursor.execute(
                "{CALL dbo.usp_DeleteByRailPeriod (?)}",
                str(rail_period)
            )
            print(f"Deleted DimDate rows for RailPeriod: {rail_period}")
            
        conn.commit()
        cursor.close()

In [ ]:
def truncate_table(table_name, WAREHOUSE_SERVER, WAREHOUSE):

    sql = f"TRUNCATE TABLE {table_name}"

    print(f"Truncating {table_name}")

    execute_sql(sql, WAREHOUSE_SERVER, WAREHOUSE)

    print(f"{table_name} truncated successfully.")

In [ ]:
def build_dim_date(silver_df):

    return (
        silver_filtered_df
        .select("date", "rail_period")
        .distinct()
        .withColumn(
            "DateKey",
            concat(
                year("date"),
                lpad(month("date"), 2, "0"),
                lpad(dayofmonth("date"), 2, "0")
            ).cast("int")
        )
        .withColumnRenamed("date", "FullDate")
        .withColumn("FullDate", to_date(col("FullDate")))
        .withColumn("CalendarYear", year("FullDate"))
        .withColumn("CalendarMonth", month("FullDate"))
        .withColumn("MonthName", date_format("FullDate", "MMMM"))
        .withColumn("QuarterNumber", quarter("FullDate"))
        .withColumn("DayOfMonth", dayofmonth("FullDate"))
        .withColumn("DayOfWeekName", date_format("FullDate", "EEEE"))
        .withColumn("RailPeriod", col("rail_period").cast("string"))
        .select(
            "DateKey",
            "FullDate",
            "CalendarYear",
            "CalendarMonth",
            "MonthName",
            "QuarterNumber",
            "DayOfMonth",
            "DayOfWeekName",
            "RailPeriod"
        )
    )

In [ ]:
def build_dim_station(silver_df):

    origin_df = silver_df.select(
        F.col("origin_of_service_crs").alias("StationCRS"),
        F.col("origin_of_service_station_name").alias("StationName")
    )

    destination_df = silver_df.select(
        F.col("destination_of_service_crs").alias("StationCRS"),
        F.col("destination_of_service_station_name").alias("StationName")
    )

    count_df = silver_df.select(
        F.col("station_crs").alias("StationCRS"),
        F.col("station_name").alias("StationName")
    )

    return (
        origin_df
        .union(destination_df)
        .union(count_df)
        .filter(F.col("StationCRS").isNotNull())
        .filter(F.col("StationName").isNotNull())
        .dropDuplicates(["StationCRS"])
    )

In [ ]:
def build_dim_operator(silver_df):

    return (
        silver_df
        .select(
            F.col("train_operator").alias("TrainOperator"),
            F.col("train_operator_brand").alias("TrainOperatorBrand")
        )
        .filter(F.col("TrainOperator").isNotNull())
        .filter(F.col("TrainOperatorBrand").isNotNull())
        .dropDuplicates()
    )

In [ ]:
def build_dim_fleet(silver_df):

    return (
        silver_df
        .select(
            F.col("fleet").cast("string").alias("FleetCode")
        )
        .filter(F.col("FleetCode").isNotNull())
        .dropDuplicates(["FleetCode"])
    )

In [ ]:
def build_dim_route(silver_df):

    return (
        silver_df
        .select(
            F.col("origin_of_service_crs").alias("OriginCRS"),
            F.col("destination_of_service_crs").alias("DestinationCRS")
        )
        .filter(F.col("OriginCRS").isNotNull())
        .filter(F.col("DestinationCRS").isNotNull())
        .dropDuplicates()
    )

In [ ]:
def build_fact_passenger_load(silver_df):

    return (
        silver_df
        .select(
            F.col("date").cast("date").alias("ServiceDate"),

            F.col("origin_of_service_crs").alias("OriginCRS"),
            F.col("destination_of_service_crs").alias("DestinationCRS"),

            F.col("station_crs").alias("CountStationCRS"),

            F.col("train_operator").alias("TrainOperator"),
            F.col("train_operator_brand").alias("TrainOperatorBrand"),

            F.col("fleet").cast("string").alias("FleetCode"),

            F.col("unique_id_uid").alias("UID"),
            F.col("train_id").alias("TrainID"),

            F.col("train_service_code").cast("string").alias("TrainServiceCode"),

            F.col("calling_point_number").cast("int").alias("CallingPointNumber"),

            F.col("number_of_cars").cast("int").alias("NumberOfCars"),

            F.col("standard_seating_capacity").cast("int")
                .alias("StandardSeatingCapacity"),

            F.col("maximum_standing_capacity").cast("int")
                .alias("MaximumStandingCapacity"),

            F.col("first_class_seating_capacity").cast("int")
                .alias("FirstClassSeatingCapacity"),

            F.col("number_of_counts").cast("int")
                .alias("NumberOfCounts"),

            F.col("total_load").cast("int")
                .alias("TotalLoad"),

            (
                F.regexp_replace(
                    F.col("total_load_percentage"),
                    "%",
                    ""
                ).cast("decimal(9,2)")
            ).alias("TotalLoadPct")
        )
    )